# Démonstration d'entraînement U-TILISE

Ce notebook montre comment lancer un entraînement du modèle U-TILISE de manière interactive.

**Contenu :**
1. Chargement et fusion de la configuration
2. Préparation des datasets (train / val) avec un sous-ensemble réduit
3. Instanciation du modèle, de l'optimiseur et du scheduler
4. Lancement de l'entraînement (quelques epochs)
5. Suivi des courbes d'entraînement via TensorBoard

> **Prérequis :** l'environnement conda `cloud_reconstruction` doit être activé.
> Un fichier HDF5 doit être accessible (chemin configuré dans le YAML).

In [ ]:
import os
import sys

# Se placer à la racine du projet
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f"Répertoire de travail : {PROJECT_ROOT}")

## 1. Configuration

La configuration est obtenue par fusion de 3 niveaux :
- `configs/default.yaml` — paramètres par défaut
- `configs/config_run_train.yaml` — surcharges d'entraînement
- Modifications manuelles ci-dessous (subset, nombre d'epochs, etc.)

In [ ]:
from omegaconf import OmegaConf
from lib import config_utils, data_utils, utils

# Charger et fusionner les configs
cfg_default = config_utils.read_config("configs/default.yaml")
cfg_train = config_utils.read_config("configs/config_run_train.yaml")
config = OmegaConf.merge(cfg_default, cfg_train)

# --- Adaptations pour la démo ---
# Sous-ensemble réduit pour un entraînement rapide
config.data.subset = 10
# Nombre d'epochs limité
config.training_settings.num_epochs = 3
# Répertoire de sortie
SAVE_DIR = "/tmp/utlise_training_demo"
config.output.output_directory = SAVE_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

print("=== Configuration (extrait) ===")
print(f"  Dataset       : {config.data.dataset}")
print(f"  HDF5          : {config.data.hdf5_file}")
print(f"  Séquence max  : {config.data.max_seq_length}")
print(f"  SAR           : {config.data.use_sar}")
print(f"  Masquage      : {config.mask.mask_type}")
print(f"  Batch size    : {config.training_settings.batch_size}")
print(f"  Epochs        : {config.training_settings.num_epochs}")
print(f"  Subset        : {config.data.subset}")
print(f"  Sortie        : {SAVE_DIR}")

## 2. Préparation des datasets et dataloaders

In [ ]:
import logging
import torch

from lib.logger import prepare_logger

# Logger simple pour la démo
logger = prepare_logger("demo", level=logging.INFO, log_to_console=True)

# Seed pour la reproductibilité
utils.set_seed(42)

# Datasets train / val
train_dset = data_utils.get_dataset(config, phase="train", logger=logger)
val_dset = data_utils.get_dataset(config, phase="val", logger=logger)

print(f"\nTaille du train set : {len(train_dset)} patches")
print(f"Taille du val set   : {len(val_dset)} patches")
print(f"Canaux              : {train_dset.num_channels}")
print(f"Taille image        : {train_dset.image_size}")

In [ ]:
# Dataloaders
subset = config.data.get("subset", False)
generator = torch.Generator()
generator.manual_seed(42)

train_loader = data_utils.get_dataloader(
    train_dset, config, drop_last=True, shuffle=True,
    generator=generator, subset=subset,
)
val_loader = data_utils.get_dataloader(
    val_dset, config, drop_last=False, shuffle=True,
    generator=generator, subset=subset,
)

print(f"Batches train : {len(train_loader)}")
print(f"Batches val   : {len(val_loader)}")

## 3. Visualisation d'un échantillon

Avant de lancer l'entraînement, visualisons un échantillon pour vérifier les données.

In [ ]:
%matplotlib inline
from lib import visutils
from lib.data_utils import extract_sample

# Charger un batch
batch = next(iter(train_loader))
inputs, target, masks, mask_valid, cloud_mask, indices_rgb, index_nir = extract_sample(batch)

print(f"Entrée (x)   : {inputs.shape}  — (B, T, C, H, W)")
print(f"Cible (y)    : {target.shape}")
print(f"Masques      : {masks.shape}")

# Galerie RGB de la cible (ground truth)
fig = visutils.sequence2gallery(
    target[0], variable="rgb",
    indices_rgb=indices_rgb.int().tolist(),
    brightness_factor=3.0,
)
fig.suptitle("Cible (ground truth) — fausses couleurs RGB", y=1.02)
fig.show()

## 4. Instanciation du modèle

In [ ]:
# Créer le répertoire expérience et sauvegarder la config
config.output.experiment_folder = utils.create_output_directory(config)
config.output.checkpoint_dir = os.path.join(config.output.experiment_folder, "checkpoints")
os.makedirs(config.output.checkpoint_dir, exist_ok=True)
config_utils.write_config(config, os.path.join(config.output.experiment_folder, "config.yaml"))

# Modèle
input_dim = train_dset.num_channels
model, args_model = utils.get_model(config, input_dim, logger)
n_params = utils.count_model_parameters(model)
print(f"\nModèle       : {config.method.model_type}")
print(f"Entrée       : {input_dim} canaux")
print(f"Paramètres   : {n_params:,}")

# Optimiseur et scheduler
optimizer = utils.get_optimizer(config, model, logger)
scheduler = utils.get_scheduler(config, optimizer, logger)

## 5. Entraînement

On utilise le `Trainer` intégré qui gère :
- La boucle train / val par epoch
- L'écriture des logs TensorBoard dans `{experiment_folder}/tb/`
- La sauvegarde des meilleurs poids

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

trainer = utils.get_trainer(
    config, train_dset, val_dset,
    train_loader, val_loader,
    model, optimizer, scheduler, device,
)

# Lancer l'entraînement
trainer.train()

## 6. Suivi via TensorBoard

Les courbes de loss et métriques sont enregistrées automatiquement dans le sous-dossier `tb/`.
On peut les visualiser directement dans le notebook :

In [ ]:
tb_log_dir = os.path.join(config.output.experiment_folder, "tb")
print(f"Répertoire TensorBoard : {tb_log_dir}")

%load_ext tensorboard
%tensorboard --logdir {tb_log_dir}

## 7. Résumé

Les fichiers générés se trouvent dans le répertoire d'expérience :

In [ ]:
print(f"Répertoire expérience : {config.output.experiment_folder}\n")
for root, dirs, files in os.walk(config.output.experiment_folder):
    level = root.replace(config.output.experiment_folder, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")

---

**Pour un entraînement complet**, utiliser le script en ligne de commande :

```bash
python run_train.py configs/config_run_train.yaml --save_dir /chemin/vers/sortie/
```

Voir le README pour les configs détaillées (architecture v4, SAR asc+desc, etc.).